In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import shapes as rio_shapes, rasterize
from shapely.geometry import shape as shp_shape
import matplotlib.pyplot as plt

# --------------------- ROOT & FOLDERS ---------------------
ROOT = Path(r"C:/Users/cespe/OneDrive_J/OneDrive/Ejercicio 7/Documents/Documents/Maestria/Paper_micro_elementos/GitHub/Change_detection_AM")

# Years / imagery
DIR_2019 = ROOT / "Data" / "Images" / "Cafine_Cafal" / "2019_sentinel"
DIR_2022 = ROOT / "Data" / "Images" / "Cafine_Cafal" / "2022"
DIR_2023 = ROOT / "Data" / "Images" / "Cafine_Cafal" / "2023"

# Shapefiles
SHP_SAL = ROOT / "Data" / "Shapefiles" / "Cafine_salt_UTM.shp"   # must contain column SAL_COL
SHP_AOI = ROOT / "Data" / "Shapefiles" / "Cafine_AOI.shp"        # AOI for area computation

# Results & figures
RES_2019 = ROOT / "Results" / "Cafine_Cafal" / "2019"
RES_2022 = ROOT / "Results" / "Cafine_Cafal" / "2022"
RES_2023 = ROOT / "Results" / "Cafine_Cafal" / "2023"
RES_CONS = ROOT / "Results" / "Cafine_Cafal"

FIG_2019 = ROOT / "Images" / "Cafine_Cafal" / "violins_2019"
FIG_2022 = ROOT / "Images" / "Cafine_Cafal" / "violins_2022"
FIG_2023 = ROOT / "Images" / "Cafine_Cafal" / "violins_2023"
FIG_FINAL = ROOT / "Images" / "Cafine_Cafal" / "violins_final"

for d in [RES_2019, RES_2022, RES_2023, RES_CONS, FIG_2019, FIG_2022, FIG_2023, FIG_FINAL]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------- PARAMETERS -------------------
NDVI_THRESHOLDS = np.arange(0.15, 0.51, 0.01)  # tau sweep
DELTA_THRESHOLD = 0.10                         # min NDVI jump (ΔNDVI)
ID_COL  = "id"
SAL_COL = "CE"
YLABEL_EC = r"EC [dS m$^{-1}$]"

# User-chosen threshold for final maps
CHOSEN_THR = 0.26  # e.g., 0.xx


In [2]:
def cliffs_delta(in_vals, out_vals):
    """
    Compute Cliff's delta (effect size) in [-1, 1].

    Positive sign means IN tends to be larger than OUT.

    Parameters
    ----------
    in_vals : array-like
        Values for the IN group.
    out_vals : array-like
        Values for the OUT group.

    Returns
    -------
    float
        Cliff's delta in [-1, 1]. NaN if any group is empty.
    """
    x = np.asarray(in_vals); x = x[np.isfinite(x)]
    y = np.asarray(out_vals); y = y[np.isfinite(y)]
    nx, ny = x.size, y.size
    if nx == 0 or ny == 0:
        return np.nan
    y_sorted = np.sort(y)
    import bisect
    greater = 0
    less = 0
    for xi in x:
        less += bisect.bisect_left(y_sorted, xi)
        greater += ny - bisect.bisect_right(y_sorted, xi)
    delta = (greater - less) / (nx * ny)
    return float(delta)


def separation_metrics(in_vals, out_vals):
    """
    Robust separation metrics between IN and OUT (no p-values).

    Parameters
    ----------
    in_vals : array-like
        Values for the IN group.
    out_vals : array-like
        Values for the OUT group.

    Returns
    -------
    dict
        Dictionary with:
        - 'med_diff': median(OUT) - median(IN)
        - 'cliffs_delta': Cliff's delta in [-1, 1]
        - 'iqr_ratio': IQR(OUT) / IQR(IN); NaN if IQR(IN) is 0 or NaN.
    """
    x = np.asarray(in_vals); x = x[np.isfinite(x)]
    y = np.asarray(out_vals); y = y[np.isfinite(y)]
    if x.size == 0 or y.size == 0:
        return dict(med_diff=np.nan, cliffs_delta=np.nan, iqr_ratio=np.nan)
    med_in, med_out = float(np.median(x)), float(np.median(y))
    iqr_in  = float(np.subtract(*np.percentile(x, [75, 25])))
    iqr_out = float(np.subtract(*np.percentile(y, [75, 25])))
    return dict(
        med_diff = med_out - med_in,
        cliffs_delta = cliffs_delta(x, y),
        iqr_ratio = (iqr_out / iqr_in) if np.isfinite(iqr_in) and iqr_in != 0 else np.nan
    )


def load_ndvi_stack_s2_2bands(folder, epsilon=1e-6):
    """
    Load an NDVI stack from Sentinel-2 images with 2 bands (B4=Red, B8=NIR).

    Parameters
    ----------
    folder : str or pathlib.Path
        Folder with GeoTIFFs named `YYYY_MM_DD.tif` (2 bands in order: B4, B8).
    epsilon : float, optional
        Small term added to denominator for numerical stability.

    Returns
    -------
    ndvi_stack : numpy.ndarray
        Array of shape (T, H, W) with NDVI (float32), sorted by date.
    dates : list of datetime.datetime
        Acquisition dates parsed from file names.
    profile : dict
        Raster profile of the first file.
    transform : affine.Affine
        Affine transform of the first file.

    Raises
    ------
    ValueError
        If no TIFFs exist or a file does not have exactly 2 bands.
    """
    files = [f for f in Path(folder).iterdir() if f.suffix.lower() in (".tif", ".tiff")]
    if not files:
        raise ValueError(f"No TIFFs in {folder}")
    dates = [datetime.strptime(f.stem, "%Y_%m_%d") for f in files]
    files = [f for _, f in sorted(zip(dates, files))]
    ndvis = []
    profile = None
    transform = None
    for f in files:
        with rasterio.open(f) as src:
            arr = src.read()
            if arr.shape[0] != 2:
                raise ValueError(f"{f.name}: expected 2 bands; found {arr.shape[0]}")
            red = arr[0].astype(np.float32)  # B4
            nir = arr[1].astype(np.float32)  # B8
            ndvi = (nir - red) / (nir + red + epsilon)
            ndvis.append(ndvi)
            if profile is None:
                profile = src.profile
                transform = src.transform
    return np.stack(ndvis), dates, profile, transform


def load_ndvi_stack_planet_4bands(folder, epsilon=1e-6):
    """
    Load an NDVI stack from Planet images with >=4 bands (band3=Red, band4=NIR).

    Parameters
    ----------
    folder : str or pathlib.Path
        Folder with GeoTIFFs named `YYYY_MM_DD.tif`, each with >=4 bands.
    epsilon : float, optional
        Small term added to denominator for numerical stability.

    Returns
    -------
    ndvi_stack : numpy.ndarray
        Array of shape (T, H, W) with NDVI (float32), sorted by date.
    dates : list of datetime.datetime
        Acquisition dates parsed from file names.
    profile : dict
        Raster profile of the first file.
    transform : affine.Affine
        Affine transform of the first file.

    Raises
    ------
    ValueError
        If no TIFFs exist or a file has < 4 bands.
    """
    files = [f for f in Path(folder).iterdir() if f.suffix.lower() in (".tif", ".tiff")]
    if not files:
        raise ValueError(f"No TIFFs in {folder}")
    dates = [datetime.strptime(f.stem, "%Y_%m_%d") for f in files]
    files = [f for _, f in sorted(zip(dates, files))]
    ndvis = []
    profile = None
    transform = None
    for f in files:
        with rasterio.open(f) as src:
            arr = src.read()
            if arr.shape[0] < 4:
                raise ValueError(f"{f.name}: expected >=4 bands; found {arr.shape[0]}")
            red = arr[2].astype(np.float32)  # band 3
            nir = arr[3].astype(np.float32)  # band 4
            ndvi = (nir - red) / (nir + red + epsilon)
            ndvis.append(ndvi)
            if profile is None:
                profile = src.profile
                transform = src.transform
    return np.stack(ndvis), dates, profile, transform


In [3]:
def detect_first_appearance(ndvi_stack, ndvi_threshold=0.3, delta_threshold=0.1):
    """
    Detect the first appearance of vegetation per pixel in an NDVI stack.

    A pixel is flagged at the first time t that satisfies:
    (NDVI_t - NDVI_{t-1}) > delta_threshold and NDVI_t > ndvi_threshold.

    Parameters
    ----------
    ndvi_stack : numpy.ndarray
        NDVI stack of shape (T, H, W) with T >= 2.
    ndvi_threshold : float, optional
        Minimum NDVI_t to consider vegetation.
    delta_threshold : float, optional
        Minimum increment between consecutive dates.

    Returns
    -------
    numpy.ndarray
        Array (H, W) float32 with the time index (1..T-1) of first appearance,
        or NaN if no appearance.
    """
    first = np.full(ndvi_stack.shape[1:], np.nan, dtype=np.float32)
    for t in range(1, ndvi_stack.shape[0]):
        prev = ndvi_stack[t - 1]
        curr = ndvi_stack[t]
        growth = (curr - prev > delta_threshold) & (curr > ndvi_threshold)
        upd = growth & np.isnan(first)
        first[upd] = t
    return first


def early_binary(ndvi_stack, thr, delta_thr):
    """
    Convert first-appearance detection into a yearly binary mask.

    Parameters
    ----------
    ndvi_stack : numpy.ndarray
        NDVI stack (T, H, W).
    thr : float
        NDVI threshold (NDVI_t > thr).
    delta_thr : float
        Minimum increment (ΔNDVI > delta_thr).

    Returns
    -------
    numpy.ndarray
        Binary mask (H, W) uint8: 1 if pixel showed early appearance at least once.
    """
    return (~np.isnan(detect_first_appearance(ndvi_stack, thr, delta_thr))).astype(np.uint8)


def save_binary_raster(arr2d, profile, transform, out_tif):
    """
    Save a 2D binary mask as single-band GeoTIFF (uint8).

    Parameters
    ----------
    arr2d : numpy.ndarray
        2D mask (H, W) or (1, H, W). Any value >0 is set to 1 (uint8).
    profile : dict
        Base raster profile (height, width, count, dtype, transform are updated).
    transform : affine.Affine
        Output affine transform.
    out_tif : str or pathlib.Path
        Output path.

    Raises
    ------
    ValueError
        If `arr2d` is not 2D nor (1, H, W).
    """
    a = np.asarray(arr2d)
    if a.ndim == 3 and a.shape[0] == 1:
        a = a[0]
    if a.ndim != 2:
        raise ValueError(f"Expected 2D or (1,H,W), got {a.shape}")
    a = (a > 0).astype("uint8")
    meta = profile.copy()
    meta.update(driver="GTiff", count=1, dtype="uint8",
                height=a.shape[0], width=a.shape[1], transform=transform)
    Path(out_tif).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(a, 1)


def build_aoi_mask(shp_aoi_path, profile, transform):
    """
    Rasterize an AOI polygon layer to the raster grid (1=inside AOI, 0=outside).

    Parameters
    ----------
    shp_aoi_path : str or pathlib.Path
        Path to the AOI shapefile.
    profile : dict
        Reference raster profile (uses 'height', 'width', 'crs').
    transform : affine.Affine
        Reference raster transform.

    Returns
    -------
    numpy.ndarray
        uint8 mask of shape (H, W): 1 inside AOI, 0 outside.

    Notes
    -----
    If the AOI CRS differs from `profile["crs"]`, it is reprojected on the fly.
    """
    shp_aoi_path = Path(shp_aoi_path)
    if not shp_aoi_path.exists():
        raise FileNotFoundError(f"AOI file not found: {shp_aoi_path}")
    aoi = gpd.read_file(shp_aoi_path)
    if aoi.empty or aoi.geometry.is_empty.all():
        raise ValueError("AOI layer is empty.")
    if aoi.crs != profile["crs"]:
        aoi = aoi.to_crs(profile["crs"])
    shapes = [(geom, 1) for geom in aoi.geometry if geom and not geom.is_empty]
    mask = rasterize(
        shapes=shapes,
        out_shape=(profile["height"], profile["width"]),
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=False
    )
    return mask


def sample_points_vs_binary(pts_gdf, bin_arr, transform, id_col=ID_COL):
    """
    Split CE values of points into IN vs OUT according to a binary mask.

    Parameters
    ----------
    pts_gdf : geopandas.GeoDataFrame
        Sampling points with 'geometry' and SAL_COL (e.g., 'CE').
        Must be in the same CRS as the raster.
    bin_arr : numpy.ndarray
        Binary mask (H, W) with 1=IN, 0=OUT.
    transform : affine.Affine
        Raster transform (to map XY to row/col).
    id_col : str, optional
        Identifier column name (not used in computation).

    Returns
    -------
    (numpy.ndarray, numpy.ndarray)
        IN values (CE) and OUT values (CE).
    """
    H, W = bin_arr.shape
    in_vals, out_vals = [], []
    inv = ~transform
    for _, row in pts_gdf.iterrows():
        c, r = inv * (row.geometry.x, row.geometry.y)
        r = int(round(r)); c = int(round(c))
        if 0 <= r < H and 0 <= c < W:
            (in_vals if bin_arr[r, c] == 1 else out_vals).append(row[SAL_COL])
    return np.array(in_vals, dtype=float), np.array(out_vals, dtype=float)


def align_to_ref(bin_arr, src_profile, ref_profile, ref_transform):
    """
    Reproject/align a binary mask to a reference grid using nearest resampling.

    Parameters
    ----------
    bin_arr : numpy.ndarray
        Source mask (H_src, W_src) uint8 (0/1).
    src_profile : dict
        Source raster profile (uses 'transform' and 'crs').
    ref_profile : dict
        Reference raster profile (uses 'height', 'width', 'crs').
    ref_transform : affine.Affine
        Reference raster transform.

    Returns
    -------
    numpy.ndarray
        Aligned mask (H_ref, W_ref) uint8.
    """
    out = np.zeros((ref_profile["height"], ref_profile["width"]), dtype=np.uint8)
    reproject(
        source=bin_arr.astype(np.uint8),
        destination=out,
        src_transform=src_profile["transform"], src_crs=src_profile["crs"],
        dst_transform=ref_transform,          dst_crs=ref_profile["crs"],
        resampling=Resampling.nearest
    )
    return (out > 0).astype(np.uint8)


In [ ]:
def violin_plot(in_vals, out_vals, title, save_path, ylabel=YLABEL_EC):
    """
    Generate a violin plot (IN vs OUT) and save as PNG.

    Parameters
    ----------
    in_vals : array-like
        CE values inside the mask (IN).
    out_vals : array-like
        CE values outside the mask (OUT).
    title : str
        Plot title.
    save_path : str or pathlib.Path
        Output PNG path.
    ylabel : str, optional
        Y-axis label.
    """
    in_vals = np.asarray(in_vals); in_vals = in_vals[np.isfinite(in_vals)]
    out_vals = np.asarray(out_vals); out_vals = out_vals[np.isfinite(out_vals)]
    if in_vals.size == 0 and out_vals.size == 0:
        print(f"[WARN] No data for violin → {save_path}")
        return
    data, positions, labels = [], [], []
    if in_vals.size > 0:
        data.append(in_vals); positions.append(1); labels.append("IN")
    if out_vals.size > 0:
        data.append(out_vals); positions.append(2 if 1 in positions else 1); labels.append("OUT")
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.violinplot(data, positions=positions, showmeans=True, showextrema=True, showmedians=True)
    ax.set_xticks(positions); ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel); ax.set_title(title)
    rng = np.random.default_rng(42)
    for vals, xpos in zip(data, positions):
        x = xpos + (rng.random(vals.size) - 0.5) * 0.25
        ax.plot(x, vals, 'o', alpha=0.45, markersize=3)
    ax.grid(True, alpha=0.2, linestyle=":")
    fig.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200); plt.close(fig)
    print(f"[VIOLIN] {save_path}")


def _build_improvement_score(df, cols=("med_diff", "cliffs_delta"), weights=(0.7, 0.3)):
    """
    Build a per-year composite score from standardized metrics.

    Parameters
    ----------
    df : pandas.DataFrame
        Must include 'year', 'tau' and the columns in `cols`.
    cols : tuple of str, optional
        Columns to combine.
    weights : tuple of float, optional
        Weights for the columns (will be normalized to sum=1).

    Returns
    -------
    pandas.DataFrame
        Copy of `df` with an added column 'improvement_score' (0–100 approx.).
    """
    out = []
    w = np.array(weights, dtype=float); w = w / np.sum(w)
    for _, sub in df.groupby("year"):
        tmp = sub.copy()
        zs = []
        for c in cols:
            v = tmp[c].astype(float).values
            mu, sd = np.nanmean(v), np.nanstd(v)
            z = (v - mu) / sd if (np.isfinite(sd) and sd > 0) else np.zeros_like(v)
            zs.append(z)
        zs = np.vstack(zs).T
        score_z = np.nansum(zs * w, axis=1)
        p5, p95 = np.nanpercentile(score_z, [5, 95])
        score = 100 * (np.clip(score_z, p5, p95) - p5) / (p95 - p5 + 1e-9)
        tmp["improvement_score"] = score
        out.append(tmp)
    return pd.concat(out, ignore_index=True)


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_sensitivity(
    all_csv_paths,
    out_png,
    tau_mark=CHOSEN_THR,
    title_size=18,
    label_size=15,
    tick_size=14,
    legend_size=15,
    ann_size=14,
    title_text="Sensitivity of separation vs. τ",
    # extras opcionales
    rescale=None,   # None -> δ en [-1,1]; 'unit' -> 0–100 via 50*(δ+1)
    use_abs=False   # True para usar |δ| (solo magnitud)
):

    """
    Sensibilidad basada SOLO en Cliff's δ (por año en líneas finas; media ±1σ en grueso).

    Parameters
    ----------
    all_csv_paths : list[str|Path]
        CSVs por año generados por `sweep_year` (deben contener 'tau' y 'cliffs_delta').
    out_png : str|Path
        PNG de salida.
    tau_mark : float
        Línea vertical marcando τ*.
    title_size, label_size, tick_size, legend_size, ann_size : int
        Tamaños de fuente para título, ejes, ticks, leyenda y anotaciones.
    rescale : None | 'unit'
        None -> se grafica δ en [-1,1].
        'unit' -> se reescala a 0–100 con 50*(δ+1) (útil para “escala 0–100”).
    use_abs : bool
        Si True, se usa |δ| en lugar de δ (magnitud de separación).
    title_text : str
        Título del gráfico ("" para sin título).
    """
    existing = [Path(p) for p in all_csv_paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError("No sensitivity CSVs found. Run sweep_year first.")

    # Cargar y quedarnos SOLO con Cliff's δ
    df = pd.concat([pd.read_csv(p) for p in existing], ignore_index=True)
    if not {"tau", "cliffs_delta"}.issubset(df.columns):
        raise ValueError("CSV(s) must include columns 'tau' and 'cliffs_delta'.")

    # Orientación: OUT > IN deseable -> δ positivo
    vals = df["cliffs_delta"].astype(float).to_numpy()
    if use_abs:
        vals = np.abs(vals)
    if rescale == "unit":
        # δ ∈ [-1,1] -> score ∈ [0,100]
        vals = 50.0 * (vals + (0 if use_abs else 1.0))
        y_label = "Cliff’s delta (scaled 0–100)"
    else:
        y_label = "Cliff’s delta (δ)"

    df = df.copy()
    df["delta_plot"] = vals

    # Curvas por año
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for y, sub in df.groupby("year"):
        sub = sub.sort_values("tau")
        ax.plot(sub["tau"], sub["delta_plot"], alpha=0.35, lw=1.5, label=y)

    # Media ±1σ por τ
    grp = df.groupby("tau")["delta_plot"]
    taus = grp.mean().index.values
    mean_curve = grp.mean().values
    std_curve  = grp.std().values
    ax.plot(taus, mean_curve, lw=2.5, label="Mean", zorder=3)
    ax.fill_between(taus, mean_curve - std_curve, mean_curve + std_curve, alpha=0.15, label="±1σ")

    # τ* marcado
    ax.axvline(float(tau_mark), linestyle="--", alpha=0.6)
    ymax = ax.get_ylim()[1]
    ax.text(float(tau_mark), ymax * 0.95, f"τ* = {float(tau_mark):.2f}",
            rotation=90, va="top", ha="right", fontsize=ann_size)

    # Estética y fuentes
    ax.set_xlabel("NDVI threshold, τ", fontsize=label_size)
    ax.set_ylabel(y_label, fontsize=label_size)
    ax.set_title(title_text, fontsize=title_size)
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.legend(loc="best", fontsize=legend_size)
    ax.tick_params(axis="both", labelsize=tick_size)

    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(f"[SENSITIVITY (Cliff)] {out_png}")


def best_tau_by_cliff(all_csv_paths, use_abs=False, rescale=None):
    """
    Devuelve el τ que maximiza la media interanual de Cliff’s δ (o |δ| si use_abs=True).
    """
    existing = [Path(p) for p in all_csv_paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError("No sensitivity CSVs found. Run sweep_year first.")
    df = pd.concat([pd.read_csv(p) for p in existing], ignore_index=True)
    if use_abs:
        df["delta_plot"] = np.abs(df["cliffs_delta"].astype(float))
    else:
        df["delta_plot"] = df["cliffs_delta"].astype(float)
    # (El rescale no cambia el argmax cuando es lineal/monótono; se ignora aquí)
    mean_by_tau = df.groupby("tau")["delta_plot"].mean()
    best_tau = float(mean_by_tau.idxmax())
    return best_tau




def plot_area_vs_tau(csv_paths, out_png, title="Classified area (%) vs NDVI threshold (τ)"):
    """
    Plot % of classified area (inside AOI) vs τ for each year and the mean.

    Parameters
    ----------
    csv_paths : list of str or Path
        Paths to `{year}_threshold_sensitivity.csv` (must contain 'area_frac').
    out_png : str or Path
        Output PNG path.
    title : str, optional
        Figure title.

    Returns
    -------
    None
    """
    existing = [Path(p) for p in csv_paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError("No sensitivity CSVs found. Run sweep_year first.")
    df = pd.concat([pd.read_csv(p) for p in existing], ignore_index=True)
    df["area_pct"] = df["area_frac"].astype(float) * 100.0

    fig, ax = plt.subplots(figsize=(7.5, 5))
    for y, sub in df.groupby("year"):
        sub = sub.sort_values("tau")
        ax.plot(sub["tau"], sub["area_pct"], lw=1.5, alpha=0.7, label=str(y))

    mean_curve = df.groupby("tau")["area_pct"].mean().sort_index()
    ax.plot(mean_curve.index.values, mean_curve.values, lw=2.5, label="Mean", zorder=3)

    ax.set_xlabel("NDVI threshold, τ")
    ax.set_ylabel("Classified area inside AOI (%)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25, linestyle=":")
    ax.legend(loc="best")
    Path(out_png).parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200); plt.close(fig)
    print(f"[AREA vs τ] {out_png}")


In [5]:
def sweep_year(year_label, year_dir, loader_fn, fig_dir, res_dir, shp_aoi=SHP_AOI):
    """
    Threshold sweep for a single year: write binary rasters, violin plots,
    and a metrics CSV (area computed inside AOI if provided).

    Parameters
    ----------
    year_label : str
        Year tag (e.g., 'Cafine_Cafal_2019').
    year_dir : str or pathlib.Path
        Folder with the imagery for that year.
    loader_fn : callable
        Function to load NDVI stack for the sensor (S2 or Planet).
        Must return (ndvi_stack, dates, profile, transform).
    fig_dir : str or pathlib.Path
        Output folder for per-threshold violin PNGs.
    res_dir : str or pathlib.Path
        Output folder for per-threshold rasters and CSV.
    shp_aoi : str or pathlib.Path, optional
        AOI polygon file; if present, area metrics are computed inside the AOI.

    Returns
    -------
    dict
        {'profile', 'transform', 'saved', 'ndvi_stack'}
    """
    ndvi_stack, dates, profile, transform = loader_fn(year_dir)

    # AOI mask (optional)
    aoi_mask = None
    total_pixels_for_area = ndvi_stack.shape[1] * ndvi_stack.shape[2]
    if shp_aoi is not None:
        try:
            aoi_mask = build_aoi_mask(shp_aoi, profile, transform)  # uint8 {0,1}
            aoi_count = int(aoi_mask.sum())
            if aoi_count > 0:
                total_pixels_for_area = aoi_count
            else:
                print("[WARN] AOI mask has zero pixels; falling back to full scene.")
                aoi_mask = None
        except Exception as e:
            print(f"[WARN] AOI not used ({e}); falling back to full scene.")
            aoi_mask = None

    # Sampling points
    pts = gpd.read_file(SHP_SAL)
    if SAL_COL not in pts.columns:
        raise ValueError(f"'{SAL_COL}' not found in {SHP_SAL}")
    if ID_COL not in pts.columns:
        pts[ID_COL] = np.arange(len(pts)) + 1
    if pts.crs != profile["crs"]:
        pts = pts.to_crs(profile["crs"])

    saved = {}
    rows = []
    for thr in NDVI_THRESHOLDS:
        bin_arr = early_binary(ndvi_stack, thr, DELTA_THRESHOLD)
        out_tif = res_dir / f"{year_label}_earlyveg_thr_{thr:.2f}.tif"
        save_binary_raster(bin_arr, profile, transform, out_tif)
        saved[float(thr)] = out_tif

        # IN/OUT CE for violin and separation metrics
        in_vals, out_vals = sample_points_vs_binary(pts, bin_arr, transform, ID_COL)
        vio_png = fig_dir / f"{year_label}_violin_thr_{thr:.2f}.png"
        violin_plot(in_vals, out_vals, f"NDVI threshold = {thr:.2f}", vio_png, ylabel=YLABEL_EC)

        # Area fraction inside AOI (if provided)
        if aoi_mask is not None:
            inside = (bin_arr.astype(bool) & aoi_mask.astype(bool)).sum()
            area_frac = float(inside) / float(total_pixels_for_area) if total_pixels_for_area > 0 else np.nan
        else:
            area_frac = float(bin_arr.sum()) / float(total_pixels_for_area)

        m = separation_metrics(in_vals, out_vals)
        rows.append({
            "year": year_label,
            "tau": float(thr),
            "n_in": int(len(in_vals)),
            "n_out": int(len(out_vals)),
            "area_frac": area_frac,  # % area is computed later as area_frac*100
            **m
        })

    df_metrics = pd.DataFrame(rows)
    csv_path = res_dir / f"{year_label}_threshold_sensitivity.csv"
    df_metrics.to_csv(csv_path, index=False)
    print(f"[METRICS] {csv_path}")

    return {"profile": profile, "transform": transform, "saved": saved, "ndvi_stack": ndvi_stack}


def apply_chosen_threshold(year_label, ndvi_stack, profile, transform, chosen_thr, out_dir):
    """
    Apply the chosen threshold to a year's NDVI stack and write the final binary raster.

    Parameters
    ----------
    year_label : str
        Year tag (e.g., 'Cafine_Cafal_2019').
    ndvi_stack : numpy.ndarray
        NDVI stack (T, H, W).
    profile : dict
        Base raster profile.
    transform : affine.Affine
        Affine transform.
    chosen_thr : float
        User-selected NDVI threshold.
    out_dir : str or pathlib.Path
        Output folder.

    Returns
    -------
    pathlib.Path
        Output path to the final GeoTIFF.
    """
    bin_arr = early_binary(ndvi_stack, chosen_thr, DELTA_THRESHOLD)
    out_final = out_dir / f"{year_label}_earlyveg_FINAL_thr_{chosen_thr:.2f}.tif"
    save_binary_raster(bin_arr, profile, transform, out_final)
    return out_final


def vectorize_and_save(mask2d, transform, crs, out_shp):
    """
    Vectorize a binary mask (1=IN) and save as Shapefile.

    Parameters
    ----------
    mask2d : numpy.ndarray
        Binary mask (H, W) with values {0,1}.
    transform : affine.Affine
        Affine transform.
    crs : dict or rasterio.crs.CRS
        Target CRS.
    out_shp : str or pathlib.Path
        Output path (.shp).
    """
    mask_arr = mask2d == 1
    feats = [
        {"geometry": shp_shape(geom), "value": int(val)}
        for geom, val in rio_shapes(mask2d.astype("uint8"), mask=mask_arr, transform=transform)
        if val == 1
    ]
    if not feats:
        print("[WARN] No polygons found (empty mask).")
        return
    gdf = gpd.GeoDataFrame(feats, geometry="geometry", crs=crs)
    gdf.to_file(out_shp)


def final_overlap(chosen_thr, ref_info, final_maps):
    """
    Intersect 3/3 yearly final maps, then write consensus raster/shape and final violin.

    Parameters
    ----------
    chosen_thr : float
        NDVI threshold used (only for naming outputs).
    ref_info : dict
        Reference info (e.g., from 2019) with 'profile' and 'transform'.
    final_maps : dict
        {year_int: Path_to_final_tif}

    Returns
    -------
    None
    """
    ref_prof = ref_info["profile"]; ref_tr = ref_info["transform"]
    aligned = []
    for year in sorted(final_maps.keys()):
        with rasterio.open(final_maps[year]) as src:
            arr = (src.read(1) > 0).astype(np.uint8)
            prof = src.profile; tr = src.transform
        if (prof["height"], prof["width"], tr) == (ref_prof["height"], ref_prof["width"], ref_tr):
            aligned.append(arr)
        else:
            aligned.append(align_to_ref(arr, prof, ref_prof, ref_tr))

    stack = np.stack(aligned, axis=0)
    cons  = (stack.sum(axis=0) == stack.shape[0]).astype(np.uint8)

    out_cons_tif = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}.tif"
    save_binary_raster(cons, ref_prof, ref_tr, out_cons_tif)
    out_cons_shp = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}.shp"
    vectorize_and_save(cons, ref_tr, ref_prof["crs"], out_cons_shp)

    # Final violin using CE points vs consensus
    pts = gpd.read_file(SHP_SAL)
    if pts.crs != ref_prof["crs"]:
        pts = pts.to_crs(ref_prof["crs"])
    in_vals, out_vals = sample_points_vs_binary(pts, cons, ref_tr, ID_COL)
    vio_final = FIG_FINAL / f"violins_final_thr_{chosen_thr:.2f}.png"
    violin_plot(in_vals, out_vals, f"Year intersection (3/3) – NDVI threshold = {chosen_thr:.2f}", vio_final, ylabel=YLABEL_EC)

    print(f"[FINAL] Raster:    {out_cons_tif}")
    print(f"[FINAL] Shapefile: {out_cons_shp}")
    print(f"[FINAL] Violin:    {vio_final}")


In [6]:
def remove_islands(gdf, output_file, min_area=1000):
    """
    Remove small polygon islands from a GeoDataFrame and dissolve into one part.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Polygon GeoDataFrame.
    output_file : str or pathlib.Path
        Output shapefile path.
    min_area : float, optional
        Minimum area to keep (units of the GeoDataFrame CRS).

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned and dissolved GeoDataFrame.
    """
    gdf = gdf.explode(index_parts=True)
    gdf = gdf[gdf.geometry.area >= min_area].reset_index(drop=True)
    gdf["value"] = 1
    gdf = gdf.dissolve(by="value", as_index=False)
    gdf.to_file(output_file, driver="ESRI Shapefile")
    return gdf[gdf.geometry.area >= min_area].reset_index(drop=True)


In [7]:
MIN_ISLAND_AREA = 1000  # area units in the consensus CRS (e.g., m² for UTM)

def rasterize_geoms_to_mask(gdf, profile, transform):
    """
    Rasterize polygon geometries to a binary mask matching a raster grid.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Polygon layer to rasterize. Must be in the same CRS as `profile`.
    profile : dict
        Reference raster profile (uses 'height', 'width', 'crs').
    transform : affine.Affine
        Reference raster transform.

    Returns
    -------
    numpy.ndarray
        uint8 mask of shape (H, W) with 1 for polygon interior, 0 otherwise.
    """
    if gdf.crs != profile["crs"]:
        gdf = gdf.to_crs(profile["crs"])
    shapes = [(geom, 1) for geom in gdf.geometry if geom and not geom.is_empty]
    mask = rasterize(
        shapes=shapes,
        out_shape=(profile["height"], profile["width"]),
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=False
    )
    return mask


def final_overlap(chosen_thr, ref_info, final_maps, min_island_area=MIN_ISLAND_AREA):
    """
    Intersect yearly final maps (3/3), enforce island removal, and write outputs.

    This function now ALWAYS removes small polygon islands from the consensus
    before the final validation/plotting. Both raw and cleaned consensus are
    written to disk (GeoTIFF + Shapefile). The final violin is computed
    against the **cleaned** consensus.

    Parameters
    ----------
    chosen_thr : float
        NDVI threshold used (only for naming outputs).
    ref_info : dict
        Reference info (e.g., from 2019) with 'profile' and 'transform'.
    final_maps : dict
        {year_int: Path_to_final_tif} mapping for the 3 years.
    min_island_area : float, optional
        Minimum polygon area to keep in the consensus (units of consensus CRS).

    Returns
    -------
    None

    Side Effects
    ------------
    Writes the following:
      - Raw consensus raster:   associated_mangrove_consensus_thr_XX.tif
      - Raw consensus shapefile:associated_mangrove_consensus_thr_XX.shp
      - CLEAN raster:           associated_mangrove_consensus_thr_XX_clean.tif
      - CLEAN shapefile:        associated_mangrove_consensus_thr_XX_clean.shp
      - Final violin (using CLEAN mask): violins_final_thr_XX.png
    """
    ref_prof = ref_info["profile"]
    ref_tr   = ref_info["transform"]

    # 1) Intersect 3/3 on the reference grid
    aligned = []
    for year in sorted(final_maps.keys()):
        with rasterio.open(final_maps[year]) as src:
            arr = (src.read(1) > 0).astype(np.uint8)
            prof = src.profile
            tr   = src.transform
        if (prof["height"], prof["width"], tr) == (ref_prof["height"], ref_prof["width"], ref_tr):
            aligned.append(arr)
        else:
            aligned.append(align_to_ref(arr, prof, ref_prof, ref_tr))
    stack = np.stack(aligned, axis=0)
    cons  = (stack.sum(axis=0) == stack.shape[0]).astype(np.uint8)

    # 2) Write RAW consensus (raster + shapefile)
    out_cons_tif = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}.tif"
    save_binary_raster(cons, ref_prof, ref_tr, out_cons_tif)

    out_cons_shp = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}.shp"
    vectorize_and_save(cons, ref_tr, ref_prof["crs"], out_cons_shp)

    # 3) Read vector, remove islands (MANDATORY), write CLEAN shapefile
    gdf_raw = gpd.read_file(out_cons_shp)
    gdf_clean = remove_islands(gdf_raw, RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}_clean.shp",
                               min_area=min_island_area)

    # 4) Rasterize CLEAN vector back to grid and write CLEAN raster
    cons_clean_mask = rasterize_geoms_to_mask(gdf_clean, ref_prof, ref_tr).astype(np.uint8)
    out_cons_clean_tif = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}_clean.tif"
    save_binary_raster(cons_clean_mask, ref_prof, ref_tr, out_cons_clean_tif)

    # 5) Final violin using the CLEAN consensus mask
    pts = gpd.read_file(SHP_SAL)
    if pts.crs != ref_prof["crs"]:
        pts = pts.to_crs(ref_prof["crs"])
    in_vals, out_vals = sample_points_vs_binary(pts, cons_clean_mask, ref_tr, ID_COL)
    vio_final = FIG_FINAL / f"violins_final_thr_{chosen_thr:.2f}.png"
    violin_plot(in_vals, out_vals, f"Year intersection (3/3) – NDVI threshold = {chosen_thr:.2f}", vio_final, ylabel=YLABEL_EC)

    print(f"[FINAL - RAW]   Raster:    {out_cons_tif}")
    print(f"[FINAL - RAW]   Shapefile: {out_cons_shp}")
    print(f"[FINAL - CLEAN] Raster:    {out_cons_clean_tif}")
    print(f"[FINAL - CLEAN] Shapefile: {gdf_clean}")
    print(f"[FINAL]         Violin:    {vio_final}")


In [8]:
def export_points_excel_and_map(chosen_thr=CHOSEN_THR):
    """
    Export EC points split by consensus polygon (IN/OUT) to Excel and show a Folium map.

    Parameters
    ----------
    chosen_thr : float, optional
        Chosen NDVI threshold τ* to locate consensus files.

    Notes
    -----
    Writes an Excel with sheets: ALL, IN_AM, OUT_AM. Displays an interactive map.
    """
    shp_am = RES_CONS / f"associated_mangrove_consensus_thr_{chosen_thr:.2f}.shp"
    shp_pts = SHP_SAL
    out_xlsx = RES_CONS / f"AM_points_summary_thr_{chosen_thr:.2f}.xlsx"

    am = gpd.read_file(shp_am)
    pts = gpd.read_file(shp_pts)

    if "CE" not in pts.columns:
        raise ValueError("Column 'CE' not found in EC points shapefile.")
    if am.crs is None:
        raise ValueError("Consensus shapefile has no CRS.")

    if pts.crs != am.crs:
        pts = pts.to_crs(am.crs)

    pts_join = gpd.sjoin(pts, am[["geometry"]], how="left", predicate="within")
    pts_join["in_am"] = pts_join.index_right.notna().astype(int)

    id_cols = [c for c in ("id", "ID", "Id") if c in pts_join.columns]
    keep_cols = id_cols + [c for c in pts.columns if c not in id_cols + ["geometry"]] + ["in_am"]
    df_all = pts_join[keep_cols].copy()
    df_in = df_all[df_all["in_am"] == 1].copy()
    df_out = df_all[df_all["in_am"] == 0].copy()

    with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as xw:
        df_all.to_excel(xw, index=False, sheet_name="ALL")
        df_in.to_excel(xw, index=False, sheet_name="IN_AM")
        df_out.to_excel(xw, index=False, sheet_name="OUT_AM")

    print(f"[OK] Excel written: {out_xlsx}")
    print(f"Total points: {len(df_all)} | IN: {len(df_in)} | OUT: {len(df_out)}")
    if len(df_in):
        print(f"EC (IN)  - median: {df_in['CE'].median():.3f}, n={len(df_in)}")
    if len(df_out):
        print(f"EC (OUT) - median: {df_out['CE'].median():.3f}, n={len(df_out)}")

    # Optional: Folium map display (for notebooks)
    try:
        import folium
        am_wgs = am.to_crs(4326)
        pts_wgs = pts_join.to_crs(4326)
        center = am_wgs.unary_union.centroid
        m = folium.Map(location=[center.y, center.x], zoom_start=13, control_scale=True, tiles="CartoDB positron")
        folium.GeoJson(
            am_wgs.__geo_interface__,
            name=f"Associated Mangrove (τ={chosen_thr:.2f})",
            style_function=lambda x: {"color": "#586f7e", "weight": 2, "fillColor": "#1f78b4", "fillOpacity": 0.25}
        ).add_to(m)

        fg_in = folium.FeatureGroup(name="IN (inside AM)", show=True)
        fg_out = folium.FeatureGroup(name="OUT (outside AM)", show=True)

        for _, r in pts_wgs.iterrows():
            color = "green" if r["in_am"] == 1 else "red"
            marker = folium.CircleMarker(
                location=[r.geometry.y, r.geometry.x],
                radius=4, color=color, fill=True, fill_opacity=0.85, weight=1
            )
            marker.add_child(folium.Popup(f"EC: {r.get('CE','NA')} | IN_AM: {int(r['in_am'])}", max_width=240))
            (fg_in if r["in_am"] == 1 else fg_out).add_child(marker)

        m.add_child(fg_in); m.add_child(fg_out)
        folium.LayerControl(collapsed=False).add_to(m)
        try:
            from IPython.display import display
            display(m)
        except Exception:
            pass
    except Exception as e:
        print(f"[WARN] Folium map not displayed ({e}).")

In [25]:
def main():
    # 1) Per-year threshold sweep (writes per-τ rasters + violins + metrics CSV)
    y2019 = sweep_year("Cafine_Cafal_2019", DIR_2019, load_ndvi_stack_s2_2bands,     FIG_2019, RES_2019, shp_aoi=SHP_AOI)
    y2022 = sweep_year("Cafine_Cafal_2022", DIR_2022, load_ndvi_stack_planet_4bands, FIG_2022, RES_2022, shp_aoi=SHP_AOI)
    y2023 = sweep_year("Cafine_Cafal_2023", DIR_2023, load_ndvi_stack_planet_4bands, FIG_2023, RES_2023, shp_aoi=SHP_AOI)
    print("\n[OK] Yearly sweeps done. Review violins and metrics, then set CHOSEN_THR.\n")

    # 2) Global plots across years (sensitivity vs τ, and % area vs τ inside AOI)
    csv_2019 = RES_2019 / "Cafine_Cafal_2019_threshold_sensitivity.csv"
    csv_2022 = RES_2022 / "Cafine_Cafal_2022_threshold_sensitivity.csv"
    csv_2023 = RES_2023 / "Cafine_Cafal_2023_threshold_sensitivity.csv"
    sens_png = FIG_FINAL / "sensitivity_vs_tau.png"
    area_png = FIG_FINAL / "area_vs_tau.png"
    _ = plot_sensitivity(
    [csv_2019, csv_2022, csv_2023],
    sens_png / "sensitivity_vs_tau.png",
    tau_mark=0.26,
    title_size=18,
    label_size=15,
    tick_size=14,
    legend_size=15,
    ann_size=14,
    rescale="unit",
    use_abs=False,
    title_text="Sensitivity of separation vs. τ"
)
   
    plot_area_vs_tau([csv_2019, csv_2022, csv_2023], area_png)
    csvs = [RES_2019/"Cafine_Cafal_2019_threshold_sensitivity.csv",
    RES_2022/"Cafine_Cafal_2022_threshold_sensitivity.csv",
    RES_2023/"Cafine_Cafal_2023_threshold_sensitivity.csv"]
    
    # Si quieres verificar el τ óptimo por δ medio:
    tau_opt = best_tau_by_cliff(csvs, use_abs=False)
    print("τ óptimo por Cliff (media interanual):", tau_opt)

    # 3) Apply the user-chosen threshold to each year → 3 final rasters
    if CHOSEN_THR is None:
        print("[ATTN] Set CHOSEN_THR (e.g., CHOSEN_THR = 0.31) and re-run.")
        return
    thr = float(CHOSEN_THR)
    f2019 = apply_chosen_threshold("Cafine_Cafal_2019", y2019["ndvi_stack"], y2019["profile"], y2019["transform"], thr, RES_2019)
    f2022 = apply_chosen_threshold("Cafine_Cafal_2022", y2022["ndvi_stack"], y2022["profile"], y2022["transform"], thr, RES_2022)
    f2023 = apply_chosen_threshold("Cafine_Cafal_2023", y2023["ndvi_stack"], y2023["profile"], y2023["transform"], thr, RES_2023)

    # 4) Intersection 3/3 and final violin
    final_overlap(thr, y2019, {2019: f2019, 2022: f2022, 2023: f2023}, min_island_area=MIN_ISLAND_AREA)
    
    # 5) Deliverables: Excel split of EC points and (optional) Folium map
    export_points_excel_and_map(chosen_thr=thr)


if __name__ == "__main__":
    main()


[VIOLIN] C:\Users\cespe\OneDrive_J\OneDrive\Ejercicio 7\Documents\Documents\Maestria\Paper_micro_elementos\GitHub\Change_detection_AM\Images\Cafine_Cafal\violins_2019\Cafine_Cafal_2019_violin_thr_0.15.png
[VIOLIN] C:\Users\cespe\OneDrive_J\OneDrive\Ejercicio 7\Documents\Documents\Maestria\Paper_micro_elementos\GitHub\Change_detection_AM\Images\Cafine_Cafal\violins_2019\Cafine_Cafal_2019_violin_thr_0.16.png
[VIOLIN] C:\Users\cespe\OneDrive_J\OneDrive\Ejercicio 7\Documents\Documents\Maestria\Paper_micro_elementos\GitHub\Change_detection_AM\Images\Cafine_Cafal\violins_2019\Cafine_Cafal_2019_violin_thr_0.17.png
[VIOLIN] C:\Users\cespe\OneDrive_J\OneDrive\Ejercicio 7\Documents\Documents\Maestria\Paper_micro_elementos\GitHub\Change_detection_AM\Images\Cafine_Cafal\violins_2019\Cafine_Cafal_2019_violin_thr_0.18.png
[VIOLIN] C:\Users\cespe\OneDrive_J\OneDrive\Ejercicio 7\Documents\Documents\Maestria\Paper_micro_elementos\GitHub\Change_detection_AM\Images\Cafine_Cafal\violins_2019\Cafine_Cafal_

C:\Users\cespe\AppData\Local\Temp\ipykernel_4452\3830153729.py:55: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = am_wgs.unary_union.centroid
